#  Employee Sentiment Analysis Project

## Final LLM Assessment

**Objective**: Analyze employee email messages to assess sentiment and engagement levels.

### Project Tasks:
1. Sentiment Labeling (Positive, Negative, Neutral)
2. Exploratory Data Analysis (EDA)
3. Employee Score Calculation (Monthly)
4. Employee Ranking
5. Flight Risk Identification
6. Predictive Modeling (Linear Regression)

---

##  Setup and Imports

First, we import all necessary libraries for data manipulation, visualization, NLP, and machine learning.

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# NLP - Sentiment Analysis
from textblob import TextBlob

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(" All libraries imported successfully!")

##  Load the Dataset

Loading the employee email dataset from the CSV file. The dataset contains email communications with the following columns:
- **Subject**: Email subject line
- **body**: Email content
- **date**: Date the email was sent
- **from**: Sender's email address (employee identifier)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/test.csv')

# Display basic information
print(f" Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\n Column Names: {list(df.columns)}")
print("\n" + "="*60)
df.head()

---

# Task 1: Sentiment Labeling 

## Objective
Label each employee message with one of three sentiment categories: **Positive**, **Negative**, or **Neutral**.

## Approach
We use **TextBlob**, a popular NLP library that provides sentiment analysis out of the box. TextBlob calculates:
- **Polarity**: Range from -1 (negative) to +1 (positive)
- **Subjectivity**: Range from 0 (objective) to 1 (subjective)

### Classification Thresholds:
- **Positive**: polarity > 0.1
- **Negative**: polarity < -0.1
- **Neutral**: -0.1 ≤ polarity ≤ 0.1

In [ ]:
# Data Preprocessing for Sentiment Analysis

# Create a copy to preserve original data
df_analysis = df.copy()

# Combine Subject and Body for more comprehensive sentiment analysis
# Fill NaN values with empty strings to avoid errors
df_analysis['text'] = df_analysis['Subject'].fillna('') + ' ' + df_analysis['body'].fillna('')

# Clean the text - remove extra whitespace
df_analysis['text'] = df_analysis['text'].str.strip()

print(f" Created combined text column")
print(f" Sample text (first 200 chars): {df_analysis['text'].iloc[0][:200]}...")

In [ ]:
def get_sentiment(text):
    """
    Analyze the sentiment of text using TextBlob.
    
    Parameters:
    -----------
    text : str
        The text to analyze
        
    Returns:
    --------
    str : 'Positive', 'Negative', or 'Neutral'
    """
    try:
        # Handle empty or non-string text
        if pd.isna(text) or str(text).strip() == '':
            return 'Neutral'
        
        # Calculate polarity using TextBlob
        polarity = TextBlob(str(text)).sentiment.polarity
        
        # Classify based on thresholds
        if polarity > 0.1:
            return 'Positive'
        elif polarity < -0.1:
            return 'Negative'
        else:
            return 'Neutral'
    except:
        return 'Neutral'

def get_polarity(text):
    """
    Get the raw polarity score for text.
    
    Returns:
    --------
    float : Polarity score between -1 and 1
    """
    try:
        if pd.isna(text) or str(text).strip() == '':
            return 0.0
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0.0

print(" Sentiment analysis functions defined")

In [ ]:
# Apply sentiment labeling to all messages
# This may take a few minutes for large datasets

print(" Analyzing sentiment for all messages... (this may take a few minutes)")

df_analysis['Sentiment'] = df_analysis['text'].apply(get_sentiment)
df_analysis['Polarity'] = df_analysis['text'].apply(get_polarity)

print("\n Sentiment labeling complete!")
print("\n Sentiment Distribution:")
print(df_analysis['Sentiment'].value_counts())
print("\n" + "="*60)
df_analysis[['Subject', 'Sentiment', 'Polarity']].head(10)

###  Observation - Task 1

The sentiment labeling has been applied to all messages in the dataset. The distribution shows the overall sentiment landscape of employee communications. We used TextBlob's polarity score with thresholds of ±0.1 to classify messages into three categories.

---

# Task 2: Exploratory Data Analysis (EDA) 

## Objective
Understand the structure, distribution, and trends in the dataset through thorough exploration.

## Analysis Areas:
1. Data structure and quality
2. Missing values analysis
3. Sentiment distribution
4. Temporal trends
5. Employee-level patterns

### 2.1 Data Structure Overview

In [ ]:
# Basic Dataset Information
print("="*60)
print(" DATASET STRUCTURE OVERVIEW")
print("="*60)

print(f"\n Total Records: {len(df_analysis):,}")
print(f" Total Columns: {len(df_analysis.columns)}")

print("\n" + "-"*40)
print("Column Data Types:")
print("-"*40)
print(df_analysis.dtypes)

In [ ]:
# Missing Values Analysis
print("\n" + "="*60)
print(" MISSING VALUES ANALYSIS")
print("="*60)

missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)

### 2.2 Date Processing and Employee Extraction

In [ ]:
# Parse dates and extract employee information

# Convert date column to datetime
df_analysis['date'] = pd.to_datetime(df_analysis['date'], format='%m/%d/%Y', errors='coerce')

# Extract employee name from email address
df_analysis['employee'] = df_analysis['from'].str.replace('@enron.com', '', regex=False)

# Extract time components
df_analysis['year'] = df_analysis['date'].dt.year
df_analysis['month'] = df_analysis['date'].dt.month
df_analysis['year_month'] = df_analysis['date'].dt.to_period('M')
df_analysis['day_of_week'] = df_analysis['date'].dt.day_name()

print(" Date parsing and employee extraction complete!")
print(f"\n Date Range: {df_analysis['date'].min()} to {df_analysis['date'].max()}")
print(f" Unique Employees: {df_analysis['employee'].nunique()}")

In [ ]:
# Display sample of processed data
df_analysis[['employee', 'date', 'year_month', 'Sentiment', 'Polarity']].head(10)

### 2.3 Sentiment Distribution Analysis

In [ ]:
# Sentiment Distribution Statistics
print("="*60)
print(" SENTIMENT DISTRIBUTION")
print("="*60)

sentiment_counts = df_analysis['Sentiment'].value_counts()
sentiment_pct = df_analysis['Sentiment'].value_counts(normalize=True) * 100

sentiment_summary = pd.DataFrame({
    'Count': sentiment_counts,
    'Percentage': sentiment_pct.round(2)
})

print(sentiment_summary)

In [ ]:
# Visualization: Sentiment Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color palette for sentiments
colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}

# Bar Chart
ax1 = axes[0]
sentiment_order = ['Positive', 'Neutral', 'Negative']
bars = ax1.bar(sentiment_order, 
               [sentiment_counts.get(s, 0) for s in sentiment_order],
               color=[colors[s] for s in sentiment_order],
               edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, label in zip(bars, sentiment_order):
    height = bar.get_height()
    ax1.annotate(f'{int(height):,}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords="offset points",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_title('Sentiment Distribution - Count', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sentiment', fontsize=12)
ax1.set_ylabel('Number of Messages', fontsize=12)

# Pie Chart
ax2 = axes[1]
ax2.pie([sentiment_counts.get(s, 0) for s in sentiment_order], 
        labels=sentiment_order,
        colors=[colors[s] for s in sentiment_order],
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.02, 0.02, 0.02),
        shadow=True)
ax2.set_title('Sentiment Distribution - Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/sentiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/sentiment_distribution.png")

###  Observation - Sentiment Distribution

The pie chart and bar graph above show the overall sentiment distribution across all employee messages. This gives us a baseline understanding of the communication tone within the organization.

### 2.4 Temporal Trends Analysis

In [ ]:
# Monthly Sentiment Trends
monthly_sentiment = df_analysis.groupby(['year_month', 'Sentiment']).size().unstack(fill_value=0)

# Ensure all sentiment columns exist
for col in ['Positive', 'Neutral', 'Negative']:
    if col not in monthly_sentiment.columns:
        monthly_sentiment[col] = 0

monthly_sentiment = monthly_sentiment[['Positive', 'Neutral', 'Negative']]

print(" Monthly Sentiment Counts (Sample):")
print(monthly_sentiment.head(10))

In [ ]:
# Visualization: Monthly Sentiment Trends
fig, ax = plt.subplots(figsize=(14, 6))

# Convert period index to string for plotting
x_labels = monthly_sentiment.index.astype(str)

ax.plot(x_labels, monthly_sentiment['Positive'], marker='o', linewidth=2, 
        label='Positive', color='#2ecc71', markersize=4)
ax.plot(x_labels, monthly_sentiment['Neutral'], marker='s', linewidth=2, 
        label='Neutral', color='#3498db', markersize=4)
ax.plot(x_labels, monthly_sentiment['Negative'], marker='^', linewidth=2, 
        label='Negative', color='#e74c3c', markersize=4)

ax.set_title('Monthly Sentiment Trends Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
ax.legend(loc='upper right', fontsize=10)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Show only every nth label to avoid crowding
n = max(1, len(x_labels) // 15)
for i, label in enumerate(ax.xaxis.get_ticklabels()):
    if i % n != 0:
        label.set_visible(False)

plt.tight_layout()
plt.savefig('../visualizations/monthly_sentiment_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/monthly_sentiment_trends.png")

###  Observation - Temporal Trends

The line chart shows how sentiment patterns change over time. Key observations:
- We can identify periods of increased positive or negative communication
- Seasonal patterns may be visible
- Spikes in negative sentiment could indicate organizational stress periods

### 2.5 Employee-Level Analysis

In [ ]:
# Top 10 Most Active Employees
employee_message_count = df_analysis['employee'].value_counts().head(10)

print(" Top 10 Most Active Employees (by message count):")
print("="*50)
for i, (emp, count) in enumerate(employee_message_count.items(), 1):
    print(f"{i:2}. {emp}: {count:,} messages")

In [ ]:
# Visualization: Top 10 Employees by Message Count with Sentiment Breakdown
top_employees = df_analysis['employee'].value_counts().head(10).index.tolist()

emp_sentiment = df_analysis[df_analysis['employee'].isin(top_employees)].groupby(
    ['employee', 'Sentiment']).size().unstack(fill_value=0)

# Ensure all columns exist
for col in ['Positive', 'Neutral', 'Negative']:
    if col not in emp_sentiment.columns:
        emp_sentiment[col] = 0

emp_sentiment = emp_sentiment[['Positive', 'Neutral', 'Negative']]
emp_sentiment = emp_sentiment.loc[top_employees]  # Maintain order

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))

emp_sentiment.plot(kind='barh', stacked=True, ax=ax,
                   color=['#2ecc71', '#3498db', '#e74c3c'],
                   edgecolor='black', linewidth=0.5)

ax.set_title('Top 10 Employees - Sentiment Breakdown', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Messages', fontsize=12)
ax.set_ylabel('Employee', fontsize=12)
ax.legend(title='Sentiment', loc='lower right')

plt.tight_layout()
plt.savefig('../visualizations/top_employees_sentiment.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/top_employees_sentiment.png")

### 2.6 Polarity Distribution Analysis

In [ ]:
# Polarity Score Distribution
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df_analysis['Polarity'], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(x=0.1, color='#2ecc71', linestyle='--', linewidth=2, label='Positive Threshold (0.1)')
ax.axvline(x=-0.1, color='#e74c3c', linestyle='--', linewidth=2, label='Negative Threshold (-0.1)')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)

ax.set_title('Distribution of Polarity Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('Polarity Score', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()

plt.tight_layout()
plt.savefig('../visualizations/polarity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n Polarity Statistics:")
print(f"   Mean: {df_analysis['Polarity'].mean():.4f}")
print(f"   Median: {df_analysis['Polarity'].median():.4f}")
print(f"   Std Dev: {df_analysis['Polarity'].std():.4f}")

###  Observation - EDA Summary

**Key Findings from EDA:**
1. The dataset contains email communications from multiple employees
2. Messages span across various time periods
3. Sentiment distribution shows the overall communication tone
4. Temporal analysis reveals trends and patterns over time
5. Employee-level analysis identifies the most active communicators

---

# Task 3: Employee Score Calculation 

## Objective
Compute a monthly sentiment score for each employee based on their messages.

## Scoring System:
- **Positive Message**: +1 point
- **Negative Message**: -1 point
- **Neutral Message**: 0 points

Scores are aggregated on a monthly basis for each employee.

In [ ]:
# Assign numerical scores to sentiments
score_mapping = {
    'Positive': 1,
    'Negative': -1,
    'Neutral': 0
}

df_analysis['Score'] = df_analysis['Sentiment'].map(score_mapping)

print(" Sentiment scores assigned")
print("\n Score Mapping:")
for sentiment, score in score_mapping.items():
    print(f"   {sentiment}: {score:+d}")

In [ ]:
# Calculate Monthly Sentiment Score for Each Employee

monthly_employee_scores = df_analysis.groupby(['employee', 'year_month']).agg({
    'Score': 'sum',
    'Sentiment': 'count'  # Total messages
}).rename(columns={'Sentiment': 'message_count'})

monthly_employee_scores = monthly_employee_scores.reset_index()
monthly_employee_scores.columns = ['employee', 'year_month', 'monthly_score', 'message_count']

print(" Monthly employee scores calculated")
print(f"\n Total Records: {len(monthly_employee_scores):,}")
print("\n" + "="*60)
print("Sample of Monthly Employee Scores:")
monthly_employee_scores.head(15)

In [ ]:
# Summary Statistics for Monthly Scores
print("="*60)
print(" MONTHLY SCORE STATISTICS")
print("="*60)

print(f"\nHighest Monthly Score: {monthly_employee_scores['monthly_score'].max()}")
print(f"Lowest Monthly Score: {monthly_employee_scores['monthly_score'].min()}")
print(f"Average Monthly Score: {monthly_employee_scores['monthly_score'].mean():.2f}")
print(f"Median Monthly Score: {monthly_employee_scores['monthly_score'].median():.2f}")

###  Observation - Employee Scores

Monthly sentiment scores have been calculated for each employee. These scores represent the net sentiment of their communications each month:
- Positive scores indicate more positive than negative messages
- Negative scores indicate more negative than positive messages
- Scores near zero suggest balanced or neutral communication

---

# Task 4: Employee Ranking 

## Objective
Generate ranked lists of employees based on their monthly sentiment scores.

## Rankings Required:
1. **Top 3 Positive Employees**: Highest positive scores
2. **Top 3 Negative Employees**: Lowest (most negative) scores

Sorting: First by score (descending for positive, ascending for negative), then alphabetically by employee name.

In [ ]:
# Get unique months in the dataset
unique_months = sorted(monthly_employee_scores['year_month'].unique())

print(f" Total unique months in dataset: {len(unique_months)}")
print(f" Date range: {unique_months[0]} to {unique_months[-1]}")

In [ ]:
def get_top_employees_for_month(month_data, top_n=3):
    """
    Get top positive and negative employees for a given month.
    
    Parameters:
    -----------
    month_data : DataFrame
        Data for a specific month
    top_n : int
        Number of top employees to return
        
    Returns:
    --------
    tuple : (top_positive_df, top_negative_df)
    """
    # Sort by score (descending) then by employee name (alphabetically)
    sorted_data = month_data.sort_values(
        by=['monthly_score', 'employee'], 
        ascending=[False, True]
    )
    
    # Top positive employees (highest scores)
    top_positive = sorted_data.head(top_n)
    
    # Top negative employees (lowest scores)
    sorted_data_neg = month_data.sort_values(
        by=['monthly_score', 'employee'], 
        ascending=[True, True]
    )
    top_negative = sorted_data_neg.head(top_n)
    
    return top_positive, top_negative

print(" Ranking function defined")

In [ ]:
# Generate rankings for all months
all_top_positive = []
all_top_negative = []

for month in unique_months:
    month_data = monthly_employee_scores[monthly_employee_scores['year_month'] == month]
    
    if len(month_data) >= 3:  # Only process months with at least 3 employees
        top_pos, top_neg = get_top_employees_for_month(month_data)
        
        top_pos = top_pos.copy()
        top_pos['rank'] = range(1, len(top_pos) + 1)
        
        top_neg = top_neg.copy()
        top_neg['rank'] = range(1, len(top_neg) + 1)
        
        all_top_positive.append(top_pos)
        all_top_negative.append(top_neg)

# Combine all months
if all_top_positive:
    top_positive_all = pd.concat(all_top_positive, ignore_index=True)
    top_negative_all = pd.concat(all_top_negative, ignore_index=True)
    
print(" Rankings generated for all months")

In [ ]:
# Display Top 3 Positive Employees (Sample - Last 5 months)
print("="*70)
print(" TOP 3 POSITIVE EMPLOYEES BY MONTH (Recent Months)")
print("="*70)

recent_months = unique_months[-5:] if len(unique_months) >= 5 else unique_months

for month in recent_months:
    month_data = top_positive_all[top_positive_all['year_month'] == month]
    if len(month_data) > 0:
        print(f"\n {month}")
        print("-"*50)
        for _, row in month_data.iterrows():
            print(f"   #{int(row['rank'])}: {row['employee']} (Score: {row['monthly_score']:+d})")

In [ ]:
# Display Top 3 Negative Employees (Sample - Last 5 months)
print("="*70)
print(" TOP 3 NEGATIVE EMPLOYEES BY MONTH (Recent Months)")
print("="*70)

for month in recent_months:
    month_data = top_negative_all[top_negative_all['year_month'] == month]
    if len(month_data) > 0:
        print(f"\n {month}")
        print("-"*50)
        for _, row in month_data.iterrows():
            print(f"   #{int(row['rank'])}: {row['employee']} (Score: {row['monthly_score']:+d})")

In [ ]:
# Overall Rankings (Aggregate across all months)
print("\n" + "="*70)
print(" OVERALL EMPLOYEE RANKINGS (Aggregate Scores)")
print("="*70)

# Calculate overall scores
overall_scores = df_analysis.groupby('employee').agg({
    'Score': 'sum',
    'Sentiment': 'count'
}).rename(columns={'Sentiment': 'total_messages', 'Score': 'total_score'})

overall_scores = overall_scores.reset_index()

# Top 3 Overall Positive
top_3_positive_overall = overall_scores.sort_values(
    by=['total_score', 'employee'], 
    ascending=[False, True]
).head(3)

# Top 3 Overall Negative
top_3_negative_overall = overall_scores.sort_values(
    by=['total_score', 'employee'], 
    ascending=[True, True]
).head(3)

print("\n TOP 3 POSITIVE EMPLOYEES (Overall):")
print("-"*50)
for i, (_, row) in enumerate(top_3_positive_overall.iterrows(), 1):
    print(f"   #{i}: {row['employee']} (Total Score: {row['total_score']:+d}, Messages: {row['total_messages']})")

print("\n TOP 3 NEGATIVE EMPLOYEES (Overall):")
print("-"*50)
for i, (_, row) in enumerate(top_3_negative_overall.iterrows(), 1):
    print(f"   #{i}: {row['employee']} (Total Score: {row['total_score']:+d}, Messages: {row['total_messages']})")

In [ ]:
# Visualization: Employee Rankings
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10 Positive Employees
top_10_positive = overall_scores.sort_values('total_score', ascending=False).head(10)
ax1 = axes[0]
bars1 = ax1.barh(top_10_positive['employee'], top_10_positive['total_score'], 
                 color='#2ecc71', edgecolor='black')
ax1.set_title('Top 10 Positive Employees\n(by Total Score)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Total Sentiment Score', fontsize=10)
ax1.invert_yaxis()

# Top 10 Negative Employees
top_10_negative = overall_scores.sort_values('total_score', ascending=True).head(10)
ax2 = axes[1]
bars2 = ax2.barh(top_10_negative['employee'], top_10_negative['total_score'], 
                 color='#e74c3c', edgecolor='black')
ax2.set_title('Top 10 Negative Employees\n(by Total Score)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Total Sentiment Score', fontsize=10)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('../visualizations/employee_rankings.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/employee_rankings.png")

###  Observation - Employee Rankings

The rankings show:
1. **Top Positive Employees**: Those with consistently positive communication patterns
2. **Top Negative Employees**: Those who may need engagement support or are expressing concerns

Rankings are sorted first by score, then alphabetically for employees with equal scores.

---

# Task 5: Flight Risk Identification 

## Objective
Identify employees who are at risk of leaving based on negative sentiment patterns.

## Criteria:
**Flight Risk**: Any employee who has sent **4 or more negative messages** within any **rolling 30-day window**.

This is a rolling count, irrespective of calendar months.

In [ ]:
# Prepare data for flight risk analysis
# Filter to only negative messages
negative_messages = df_analysis[df_analysis['Sentiment'] == 'Negative'].copy()
negative_messages = negative_messages.sort_values(['employee', 'date'])

print(f" Total Negative Messages: {len(negative_messages):,}")
print(f" Employees with Negative Messages: {negative_messages['employee'].nunique()}")

In [ ]:
def identify_flight_risks(df, threshold=4, window_days=30):
    """
    Identify employees who sent 4+ negative messages in any 30-day rolling window.
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame containing negative messages with 'employee' and 'date' columns
    threshold : int
        Minimum number of negative messages to be considered flight risk
    window_days : int
        Size of the rolling window in days
        
    Returns:
    --------
    set : Set of employee names flagged as flight risks
    """
    flight_risk_employees = set()
    
    for employee in df['employee'].unique():
        emp_messages = df[df['employee'] == employee].copy()
        emp_messages = emp_messages.sort_values('date')
        
        if len(emp_messages) < threshold:
            continue
        
        dates = emp_messages['date'].tolist()
        
        # Check each message as a potential window start
        for i, start_date in enumerate(dates):
            end_date = start_date + timedelta(days=window_days)
            
            # Count messages in this window
            messages_in_window = sum(1 for d in dates if start_date <= d <= end_date)
            
            if messages_in_window >= threshold:
                flight_risk_employees.add(employee)
                break  # No need to check further windows for this employee
    
    return flight_risk_employees

print(" Flight risk identification function defined")

In [ ]:
# Identify Flight Risk Employees
print(" Identifying flight risk employees...")

flight_risks = identify_flight_risks(negative_messages, threshold=4, window_days=30)

print(f"\n FLIGHT RISK EMPLOYEES IDENTIFIED: {len(flight_risks)}")
print("="*60)

In [ ]:
# Display Flight Risk Employees with Details
flight_risk_list = sorted(list(flight_risks))

if len(flight_risk_list) > 0:
    print("\n LIST OF FLIGHT RISK EMPLOYEES:")
    print("-"*60)
    
    flight_risk_details = []
    
    for emp in flight_risk_list:
        emp_neg = negative_messages[negative_messages['employee'] == emp]
        total_neg = len(emp_neg)
        
        flight_risk_details.append({
            'Employee': emp,
            'Total Negative Messages': total_neg,
            'First Negative': emp_neg['date'].min(),
            'Last Negative': emp_neg['date'].max()
        })
    
    flight_risk_df = pd.DataFrame(flight_risk_details)
    flight_risk_df = flight_risk_df.sort_values('Total Negative Messages', ascending=False)
    
    print(flight_risk_df.to_string(index=False))
else:
    print("\n No flight risk employees identified based on the criteria.")

In [ ]:
# Visualization: Flight Risk Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: Flight Risk vs Non-Flight Risk
ax1 = axes[0]
total_employees = df_analysis['employee'].nunique()
flight_risk_count = len(flight_risks)
non_flight_risk = total_employees - flight_risk_count

ax1.pie([flight_risk_count, non_flight_risk], 
        labels=['Flight Risk', 'Not at Risk'],
        colors=['#e74c3c', '#2ecc71'],
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.05, 0))
ax1.set_title(f'Flight Risk Distribution\n({flight_risk_count} out of {total_employees} employees)', 
              fontsize=12, fontweight='bold')

# Bar chart: Top 10 Flight Risk Employees by Negative Message Count
ax2 = axes[1]
if len(flight_risk_df) > 0:
    top_flight_risk = flight_risk_df.head(10)
    ax2.barh(top_flight_risk['Employee'], top_flight_risk['Total Negative Messages'],
             color='#e74c3c', edgecolor='black')
    ax2.set_title('Top 10 Flight Risk Employees\n(by Negative Message Count)', 
                  fontsize=12, fontweight='bold')
    ax2.set_xlabel('Number of Negative Messages', fontsize=10)
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, 'No Flight Risk Employees', ha='center', va='center', fontsize=14)
    ax2.set_title('Flight Risk Employees', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/flight_risk_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/flight_risk_analysis.png")

###  Observation - Flight Risk Identification

**Flight Risk Criteria Applied:**
- Employees who sent 4 or more negative messages within any rolling 30-day period
- This identifies employees who may be experiencing sustained negative sentiment

**Recommendations:**
- These employees should be prioritized for engagement initiatives
- Consider one-on-one check-ins to understand their concerns
- Review their recent communications for specific issues

---

# Task 6: Predictive Modeling (Linear Regression) 

## Objective
Develop a linear regression model to analyze sentiment trends and predict sentiment scores.

## Features Used:
- Message frequency (count per month)
- Average message length
- Word count
- Other derived features

In [ ]:
# Feature Engineering for Predictive Modeling

# Calculate text-based features
df_analysis['message_length'] = df_analysis['text'].str.len()
df_analysis['word_count'] = df_analysis['text'].str.split().str.len()

print(" Text features calculated")
print(f"   Average Message Length: {df_analysis['message_length'].mean():.2f} characters")
print(f"   Average Word Count: {df_analysis['word_count'].mean():.2f} words")

In [ ]:
# Aggregate features by employee and month for modeling
modeling_data = df_analysis.groupby(['employee', 'year_month']).agg({
    'Score': 'sum',  # Target variable (monthly sentiment score)
    'text': 'count',  # Message count
    'message_length': 'mean',  # Average message length
    'word_count': 'mean',  # Average word count
    'Polarity': 'mean'  # Average polarity
}).reset_index()

modeling_data.columns = ['employee', 'year_month', 'monthly_score', 'message_count', 
                         'avg_message_length', 'avg_word_count', 'avg_polarity']

# Remove any rows with missing values
modeling_data = modeling_data.dropna()

print(f" Modeling Dataset Shape: {modeling_data.shape}")
modeling_data.head()

In [ ]:
# Define features (X) and target (y)
feature_columns = ['message_count', 'avg_message_length', 'avg_word_count']

X = modeling_data[feature_columns]
y = modeling_data['monthly_score']

print(" Feature Summary:")
print(X.describe())

print(f"\n Target Variable (monthly_score):")
print(f"   Mean: {y.mean():.2f}")
print(f"   Std: {y.std():.2f}")
print(f"   Range: {y.min()} to {y.max()}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f" Training Set Size: {len(X_train)}")
print(f" Testing Set Size: {len(X_test)}")

In [ ]:
# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

print(" Linear Regression Model Trained")
print("\n Model Coefficients:")
print("-"*50)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"   {feature}: {coef:.4f}")

print(f"\n   Intercept: {model.intercept_:.4f}")

In [ ]:
# Make Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate Evaluation Metrics
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print("="*60)
print(" MODEL EVALUATION METRICS")
print("="*60)

print("\n Training Set Performance:")
print(f"   R² Score: {train_r2:.4f}")
print(f"   MSE: {train_mse:.4f}")
print(f"   MAE: {train_mae:.4f}")
print(f"   RMSE: {np.sqrt(train_mse):.4f}")

print("\n Testing Set Performance:")
print(f"   R² Score: {test_r2:.4f}")
print(f"   MSE: {test_mse:.4f}")
print(f"   MAE: {test_mae:.4f}")
print(f"   RMSE: {np.sqrt(test_mse):.4f}")

In [ ]:
# Visualization: Model Performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted (Test Set)
ax1 = axes[0]
ax1.scatter(y_test, y_pred_test, alpha=0.5, color='#3498db', edgecolor='black', linewidth=0.5)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Monthly Score', fontsize=11)
ax1.set_ylabel('Predicted Monthly Score', fontsize=11)
ax1.set_title(f'Actual vs Predicted\n(Test Set, R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
ax1.legend()

# Residuals Distribution
ax2 = axes[1]
residuals = y_test - y_pred_test
ax2.hist(residuals, bins=30, color='#9b59b6', edgecolor='black', alpha=0.7)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residuals (Actual - Predicted)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of Residuals', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/model_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/model_performance.png")

In [ ]:
# Feature Importance Visualization
fig, ax = plt.subplots(figsize=(10, 5))

importance = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=True)

colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in importance['Coefficient']]

ax.barh(importance['Feature'], importance['Coefficient'], color=colors, edgecolor='black')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.set_title('Feature Coefficients in Linear Regression Model', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Visualization saved: visualizations/feature_importance.png")

###  Observation - Predictive Modeling

**Model Summary:**
- Used Linear Regression to predict monthly sentiment scores
- Features: message count, average message length, average word count

**Interpretation of Coefficients:**
- Positive coefficients indicate features that increase sentiment scores
- Negative coefficients indicate features that decrease sentiment scores

**Model Performance:**
- R² score indicates the proportion of variance explained by the model
- Lower MSE/MAE values indicate better prediction accuracy

---

#  Summary and Key Findings

## Project Completion Summary

In [ ]:
# Final Summary
print("="*70)
print(" EMPLOYEE SENTIMENT ANALYSIS - FINAL SUMMARY")
print("="*70)

print("\n DATASET OVERVIEW:")
print(f"   Total Messages Analyzed: {len(df_analysis):,}")
print(f"   Unique Employees: {df_analysis['employee'].nunique()}")
print(f"   Date Range: {df_analysis['date'].min().date()} to {df_analysis['date'].max().date()}")

print("\n SENTIMENT DISTRIBUTION:")
for sentiment in ['Positive', 'Neutral', 'Negative']:
    count = len(df_analysis[df_analysis['Sentiment'] == sentiment])
    pct = count / len(df_analysis) * 100
    print(f"   {sentiment}: {count:,} ({pct:.1f}%)")

print("\n TOP 3 POSITIVE EMPLOYEES (Overall):")
for i, (_, row) in enumerate(top_3_positive_overall.iterrows(), 1):
    print(f"   {i}. {row['employee']} (Score: {row['total_score']:+d})")

print("\n TOP 3 NEGATIVE EMPLOYEES (Overall):")
for i, (_, row) in enumerate(top_3_negative_overall.iterrows(), 1):
    print(f"   {i}. {row['employee']} (Score: {row['total_score']:+d})")

print(f"\n FLIGHT RISK EMPLOYEES: {len(flight_risks)}")
if len(flight_risk_list) > 0:
    print(f"   List: {', '.join(flight_risk_list[:10])}")
    if len(flight_risk_list) > 10:
        print(f"   ... and {len(flight_risk_list) - 10} more")

print(f"\n PREDICTIVE MODEL PERFORMANCE:")
print(f"   R² Score (Test): {test_r2:.4f}")
print(f"   RMSE (Test): {np.sqrt(test_mse):.4f}")

print("\n" + "="*70)
print(" ANALYSIS COMPLETE")
print("="*70)

In [ ]:
# Save processed data for reference
df_analysis.to_csv('../data/processed_sentiment_data.csv', index=False)
monthly_employee_scores.to_csv('../data/monthly_employee_scores.csv', index=False)

# Save flight risk list
if len(flight_risk_df) > 0:
    flight_risk_df.to_csv('../data/flight_risk_employees.csv', index=False)

print(" Processed data saved to data/ folder")

---

##  Recommendations

Based on the analysis, here are key recommendations:

1. **Employee Engagement**: Focus on employees identified as flight risks for immediate engagement initiatives

2. **Positive Reinforcement**: Recognize top positive employees as cultural ambassadors

3. **Trend Monitoring**: Implement regular sentiment tracking to identify emerging issues early

4. **Communication Training**: Consider training for employees with consistently negative sentiment patterns

5. **Further Analysis**: Investigate specific topics or events that correlate with sentiment changes

---

*Analysis completed using Python, TextBlob, and Scikit-learn*